In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv(
    "../data/raw/FlightInfo.csv",
    sep="|", #the separator is a pipe character
    engine="python" # Use the Python engine to handle the separator
                 )
# Drop the first column which is unnamed
df.drop(columns=[df.columns[0]], inplace=True) 
df.head()

,SEQ_NBR,SEQ_SCHD_START_DT,FLEET,BASE,DIVISION,SPOILAGE,TOTAL_BLOCKED_HRS,TOTAL_SPOILED_HRS,SEQ_CAL_DAYS,SEQ_DUTY_DAYS,...,MAX_LEGS_PER_DAY,SEQ_TTL_LEGS,MORETHAN2_321_LEGS,IN_SEQ_DHD,LAYOVER,SEQ_PATTERN,SEQ_START,FLIGHT_PATTERN,SEQ_START_HRS,SF_LOAD_TMS
0,4062,2025-02-06,320,CLT,I,NOT SPOILED,NaN,NaN,2,2,...,3,6,1,0,12,CLT-CUN-CLT-RDU-CLT-MCO-CLT,Starts10:25,1265-1210-1645-3026-3126-3126,10,2025-11-06 21:57:37.430
1,195,2024-04-17,777,DFW,I,NOT SPOILED,19.42,NaN,3,2,...,1,2,0,0,25,DFW-LHR-DFW,Starts15:15,50-51,15,2025-11-06 21:57:37.430
2,26551,2025-05-11,737,LAX,D,NOT SPOILED,NaN,NaN,3,3,...,2,6,0,0,15,SNA-DFW-ABQ-DFW-RNO-DFW-SNA,Starts7:45,1382-9-1035-324-1361-2465,7,2025-11-06 21:57:37.430
3,18556,2024-12-31,737,DFW,I,NOT SPOILED,NaN,NaN,1,1,...,2,2,0,0,1,DFW-NAS-DFW,Starts10:0,2585-2585,10,2025-11-06 21:57:37.430
4,20413,2024-07-06,737,MIA,I,NOT SPOILED,5.50,NaN,1,1,...,2,2,0,0,1,MIA-GUA-MIA,Starts11:11,2241-1258,11,2025-11-06 21:57:37.430


## Defining "Change in Arrival"

The target variable is the difference between the arrival prediction at takeoff and the final arrival time, measured in minutes.

## Goal
We want the problem to be framed as regression task predicting the minute-level shift between the arrival estimate at takeoff and the final arrival time, allowing us to model not just whether or not predictions change, but by how much and in which direction.





In [ ]:
# Checking the structure of the dataset.
##df.columns
##df.info()

# Identifiying missing values in the dataset. 
##df.isna().mean().sort_values(ascending=False)

# for Blocked & Spoiled hours (if theres none the value is empty instead of 0)
#Fixing that
df["TOTAL_SPOILED_HRS"]= df["TOTAL_SPOILED_HRS"].fillna(0, inplace=False)
df["TOTAL_BLOCKED_HRS"]= df["TOTAL_BLOCKED_HRS"].fillna(0, inplace=False)


df.isna().mean().sort_values(ascending=False)
df.describe()


,SEQ_NBR,FLEET,TOTAL_BLOCKED_HRS,TOTAL_SPOILED_HRS,SEQ_CAL_DAYS,SEQ_DUTY_DAYS,SEQ_TTL_FLTTIME,MIN_FLYTIME_PER_LEG,MAX_LEGS_PER_DAY,SEQ_TTL_LEGS,MORETHAN2_321_LEGS,IN_SEQ_DHD,LAYOVER,SEQ_START_HRS
count,9025.000000,9025.000000,9025.000000,9025.000000,9025.000000,9025.000000,9025.000000,9025.000000,9025.000000,9025.000000,9025.000000,9025.000000,9025.000000,9025.000000
mean,11794.947258,595.446205,6.604647,2.895917,2.243102,2.019058,12.948476,222.685651,2.211080,3.954017,0.007091,0.035346,14.521884,11.290637
std,8883.246913,208.800611,7.885751,5.448079,0.859387,0.712316,6.072470,197.870008,0.813566,2.073343,0.083916,0.233907,8.840025,3.932451
min,101.000000,320.000000,0.000000,0.000000,1.000000,1.000000,2.000000,45.000000,1.000000,2.000000,0.000000,0.000000,0.000000,5.000000
25%,3860.000000,320.000000,0.000000,0.000000,2.000000,2.000000,10.000000,89.000000,2.000000,2.000000,0.000000,0.000000,11.000000,8.000000
50%,9263.000000,737.000000,0.000000,0.000000,2.000000,2.000000,11.000000,145.000000,2.000000,4.000000,0.000000,0.000000,16.000000,10.000000
75%,19771.000000,737.000000,11.950000,3.920000,3.000000,2.000000,16.000000,215.000000,3.000000,6.000000,0.000000,0.000000,21.000000,14.000000
max,28753.000000,787.000000,39.660000,39.650000,5.000000,4.000000,39.000000,845.000000,4.000000,11.000000,1.000000,2.000000,49.000000,23.000000


In [20]:
df.head()

,SEQ_NBR,SEQ_SCHD_START_DT,FLEET,BASE,DIVISION,SPOILAGE,TOTAL_BLOCKED_HRS,TOTAL_SPOILED_HRS,SEQ_CAL_DAYS,SEQ_DUTY_DAYS,...,MAX_LEGS_PER_DAY,SEQ_TTL_LEGS,MORETHAN2_321_LEGS,IN_SEQ_DHD,LAYOVER,SEQ_PATTERN,SEQ_START,FLIGHT_PATTERN,SEQ_START_HRS,SF_LOAD_TMS
0,4062,2025-02-06,320,CLT,I,NOT SPOILED,0.00,0.0,2,2,...,3,6,1,0,12,CLT-CUN-CLT-RDU-CLT-MCO-CLT,Starts10:25,1265-1210-1645-3026-3126-3126,10,2025-11-06 21:57:37.430
1,195,2024-04-17,777,DFW,I,NOT SPOILED,19.42,0.0,3,2,...,1,2,0,0,25,DFW-LHR-DFW,Starts15:15,50-51,15,2025-11-06 21:57:37.430
2,26551,2025-05-11,737,LAX,D,NOT SPOILED,0.00,0.0,3,3,...,2,6,0,0,15,SNA-DFW-ABQ-DFW-RNO-DFW-SNA,Starts7:45,1382-9-1035-324-1361-2465,7,2025-11-06 21:57:37.430
3,18556,2024-12-31,737,DFW,I,NOT SPOILED,0.00,0.0,1,1,...,2,2,0,0,1,DFW-NAS-DFW,Starts10:0,2585-2585,10,2025-11-06 21:57:37.430
4,20413,2024-07-06,737,MIA,I,NOT SPOILED,5.50,0.0,1,1,...,2,2,0,0,1,MIA-GUA-MIA,Starts11:11,2241-1258,11,2025-11-06 21:57:37.430
